In [ ]:
################################################################################

# Config

RUNTIME_TYPE = 'COLAB' #@param ["COLAB", "CONDA"]
MODEL_NAME = 't5-base' #@param ["t5-base", "google/flan-t5-xl", "t5-small", "t5-base", "t5-large", "google/flan-t5-small", "google/flan-t5-base", "google/flan-t5-large"]
LEARNING_RATE = 3e-4 #@param {"type": "number"}
WD = 0.0 #@param {"type": "number"}
STUFF_COUNT = 5 #@param {"type": "number"}
MAX_SOURCES_COUNT = 1 #@param {"type": "number"}
PREDICT_WITH_GENERATE = False #@param {type:"boolean"}
DATASET_NAME = "TREC" #@param ["SST1", "SST2", "MR", "SUBJ", "CR", "MPQA", "TREC", "MPQA-P", "IMDB"]
MAX_LENGTHS_ARRAY = {"SST1": 256, "SST2": 256, "MR": 104,
                     "SUBJ": 217, "CR": 142, "MPQA": 58,
                     "TREC": 54}

SEED = 0 #@param {"type": "number"}
FOLDS = 10 #@param {"type": "number"}
MULTI_STEP_FOLDED = True #@param {type:"boolean"}
STEP_FOLDS = [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]]

# if DATASET_NAME in ["TREC"]:
#   STEP_FOLDS = [[1, 2, 3, 4, 5], [6, 7, 8, 9, 10]]
# else:
#   STEP_FOLDS = [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]] # MPQA
STEP_INDICATOR = 0 #@param {"type": "number"}
HAS_TEST = ["SST1", "SST2", "TREC"]
NUM_TRAIN_EPOCHS = 5 #@param {"type": "number"}
EVALUATE = [True]
GENERATE = [True]
DO_AUGMENTATION = False #@param {type:"boolean"}
ENABLE_AUG = [DO_AUGMENTATION]
EXPERIMENT_NAME = 'prompt-aug' #@param {"type": "string"}
SAVED_ITEM_KEY = 'saved'
if MULTI_STEP_FOLDED:
    SAVED_FILES = [
        [
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>'
        ],
        [
            '<GOOGLE_DRIVE_URL_PLACEHOLDER>'
        ]
    ]
else:
    SAVED_FILES = [
        '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
        '<GOOGLE_DRIVE_URL_PLACEHOLDER>'
    ]
PROMPT_TOKEN_INDEX = 3
GENERATE_MAX_LEN = 128
# Define special tokens for putting in the input sentence (prefix)
PREFIX = ['answer: ' ]

FP16 = False
LOCAL_RANK = -1
FP16_OPT_LEVEL = 'O1'
BATCH_SIZE = 16
TRAIN_BATCH_SIZE = BATCH_SIZE
VAL_BATCH_SIZE = BATCH_SIZE
TEST_BATCH_SIZE = BATCH_SIZE
DROPOUT = 0.1
SEED = 0
DATA = 'MPQA2.0_v221219_cleaned'
IDS = 'MPQA2.0_v221219_cleaned_Source_Span_IDs'
READ_DATA_ONLINE = False
K_FOLD = 5
SST2_DICT = {0: 'negative', 1: 'positive'}
IMDB_DICT = {0: 'negative', 1: 'positive'}
TREC_DICT = {0: 'Abbreviation', 1: 'Entity',
             2: 'Description', 3: 'Human being',
             4: 'Location', 5: 'Numeric value'}

SST2_COL_MAP = {'input': 'sentence', 'output': 'label'}
IMDB_COL_MAP = {'input': 'text', 'output': 'sentiment'}
TREC_COL_MAP = {'input': 'text', 'output': 'label'}


BOUND_SUBSET = [500] #BATCH_SIZE


INIT_W = [None]
BEST_W = [None]
BEST_LOSS = [float('inf')]

INDICATOR = [0, 0]

INDICES_OUT = [list(), 0]
COUNTER_SUBSET = [0]
AUG_STATUS = [False]

INDICATOR = [0, 0]

INDICES_OUT = [list()]


NUM_BEAMS = 2
LENGTH_PENALTY = 1.0
REPITITION_PENALTY = 2.5
DO_SAMPLE = False

MORE_THAN_ONE = {'exp': 0, 'target': 0, 'agent': 0}
TOTAL_COUNT = {'exp': 0, 'target': 0, 'agent': 0}


TOKENIZER = [None]
COUNTER = [0, 0]
SENTS_LENGHTS = dict()

ERRORS = []
ONCE_DONE = [False, False]
ACTUAL_TRIPLETS = [{}, {}]
GOLD_NUM = [None, None]

EVALUATION_CRI = [-1, 3]

WRITE_FILE = [False]

In [ ]:
# Determine evaluation function
EVALUATION_METHOD = 'SIMPLE'


In [ ]:
!pip install datasets --quiet

In [ ]:
import os
from os import chdir
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
import re
import random
import numpy as np
import torch
from nltk import word_tokenize
import nltk
from nltk.corpus import stopwords
from sklearn.metrics import f1_score
import re
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import io
import transformers
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback, AutoConfig
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import json
from urllib.request import urlopen
import urllib
import seaborn as sns
import statistics
from tqdm import tqdm
from torch import optim
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from datetime import datetime
import random
import time
from datetime import date
from datasets import load_dataset

from nltk.tokenize.treebank import TreebankWordDetokenizer
detokenizer = TreebankWordDetokenizer()

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [ ]:

if RUNTIME_TYPE == 'COLAB':
  from google.colab import output
  output.enable_custom_widget_manager()
  from google.colab import drive
  drive.mount('/content/drive')
  if not os.path.exists(f'drive/MyDrive/{EXPERIMENT_NAME}'):
    os.makedirs(f'drive/MyDrive/{EXPERIMENT_NAME}')
  chdir(f'drive/MyDrive/{EXPERIMENT_NAME}')
  # os.system('rm -rf *')
else:
  if not os.path.exists(f'{EXPERIMENT_NAME}'):
    os.makedirs(f'{EXPERIMENT_NAME}')
  chdir(f'{EXPERIMENT_NAME}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content/drive/MyDrive/prompt-aug


In [ ]:
!nvidia-smi

Sun Aug 24 07:03:59 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# To assure deterministic results
import os
from os import chdir

os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ":4096:8"

In [ ]:
# Start timer
start_time = datetime.now()

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'There are {torch.cuda.device_count()} GPU(s) available.')
    print('Device name:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")


There are 1 GPU(s) available.
Device name: Tesla T4


TPI config

In [ ]:
TYPE_IDS = ['agreement', 'argue', 'intention', 'sentiment']
POLARITY_IDS = ['negative', 'positive']
INTENSITY_IDS = ['slight', 'low', 'medium', 'high', 'extreme']

In [ ]:
# TPI Classes

TYPE_CLASSES = ['agreement', 'arguing', 'intention', 'sentiment']
POLARITY_CLASSES = ['negative', 'positive']
INTENSITY_CLASSES = ['low', 'low medium', 'medium', 'medium high', 'high']
NUM_TYPE_CLASSES = len(TYPE_CLASSES)
NUM_POLARITY_CLASSES = len(POLARITY_CLASSES)
NUM_INTENSITY_CLASSES = 5
NUM_INTENSITY_UNITS = 3

if DATASET_NAME == 'MPQA-T':
    CLASSES = TYPE_CLASSES
    NUM_CLASSES = NUM_TYPE_CLASSES
elif DATASET_NAME == 'MPQA-P':
    CLASSES = POLARITY_CLASSES
    NUM_CLASSES = NUM_POLARITY_CLASSES
elif DATASET_NAME == 'MPQA-I':
    CLASSES = INTENSITY_CLASSES
    NUM_CLASSES = NUM_INTENSITY_CLASSES
else:
    CLASSES = []
    NUM_CLASSES = 0

In [ ]:
# Create a map for class ids and class names

type_classname2classindex = {
    'agreement': 0,
    'arguing':   1,
    'intention': 2,
    'sentiment': 3,
}
type_classname2classid = {
    'agreement': TYPE_IDS[0],
    'arguing':   TYPE_IDS[1],
    'intention': TYPE_IDS[2],
    'sentiment': TYPE_IDS[3],
}
type_classid2classname = {v:k for k, v in type_classname2classid.items()}
type_classid2classindex = {type_classname2classid[k]:v for k, v in type_classname2classindex.items()}

polarity_classname2classindex = {
    'negative': 0,
    'positive': 1,
}
polarity_classname2classid = {
    'negative': POLARITY_IDS[0],
    'positive': POLARITY_IDS[1],
}
polarity_classid2classname = {v:k for k, v in polarity_classname2classid.items()}
polarity_classid2classindex = {polarity_classname2classid[k]:v for k, v in polarity_classname2classindex.items()}

intensity_classname2classindices = {
    'low':        [0],
    'low medium': [0, 1],
    'medium':        [1],
    'medium high':   [1, 2],
    'high':             [2],
}
intensity_classname2classid = {
    'low':         INTENSITY_IDS[0],
    'low medium':  INTENSITY_IDS[1],
    'medium':      INTENSITY_IDS[2],
    'medium high': INTENSITY_IDS[3],
    'high':        INTENSITY_IDS[4],
}
intensity_classid2classname = {v:k for k, v in intensity_classname2classid.items()}
intensity_classid2classindices = {intensity_classname2classid[k]:v for k, v in intensity_classname2classindices.items()}

In [ ]:
import time
def pd_read_csv(link):
    t = 15
    while True:
        try:
            return pd.read_csv(link)
        except Exception as e:
            print(f"Unsuccessful attempt to download {link}. Waiting for {t}s.")
            time.sleep(t)
            t *= random.random()+1
            t = int(t)
            continue

In [ ]:
# Decompose X and y

def decompose_e2e(dataset):
    X = [i for i in dataset.mr]
    y = [i for i in dataset.ref]
    return X, y

def decompose_mpqa(dataset):
    X = [i for i in dataset.input]
    if DATASET_NAME == 'MPQA-T':
        y = [i.replace(TYPE_CLASSES[0], TYPE_IDS[0]) \
              .replace(TYPE_CLASSES[1], TYPE_IDS[1]) \
              .replace(TYPE_CLASSES[2], TYPE_IDS[2]) \
              .replace(TYPE_CLASSES[3], TYPE_IDS[3]) for i in dataset.output]
    elif DATASET_NAME == 'MPQA-P':
        y = [i.replace(POLARITY_CLASSES[0], POLARITY_IDS[0]) \
              .replace(POLARITY_CLASSES[1], POLARITY_IDS[1]) for i in dataset.output]
    elif DATASET_NAME == 'MPQA-I':
        # Add special characters to avoid conflicts
        y = [('!'+i+'!').replace('!'+INTENSITY_CLASSES[0]+'!', INTENSITY_IDS[0]) \
                        .replace('!'+INTENSITY_CLASSES[1]+'!', INTENSITY_IDS[1]) \
                        .replace('!'+INTENSITY_CLASSES[2]+'!', INTENSITY_IDS[2]) \
                        .replace('!'+INTENSITY_CLASSES[3]+'!', INTENSITY_IDS[3]) \
                        .replace('!'+INTENSITY_CLASSES[4]+'!', INTENSITY_IDS[4]) for i in dataset.output]
    return X, y

In [ ]:
# MPQA links
mpqa_t_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_train.csv'
mpqa_t_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_val.csv'
mpqa_t_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/t_test.csv'

mpqa_p_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_train.csv'
mpqa_p_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_val.csv'
mpqa_p_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/p_test.csv'

mpqa_i_train_link = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_train.csv'
mpqa_i_val_link   = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_val.csv'
mpqa_i_test_link  = 'https://github.com/gu-sentiment-2021/sent/raw/main/summer21/dataset/afl-tpi/i_test.csv'

In [ ]:
# Dataset public URL

data_name_to_google_drive_url = {
    'MPQA2.0_v221219_cleaned': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'MPQA2.0_v221219_cleaned_Source_Span_IDs': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '0.aaai19srl.train0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '0.aaai19srl.dev0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '0.aaai19srl.test0.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '1.aaai19srl.train1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '1.aaai19srl.dev1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '1.aaai19srl.test1.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '2.aaai19srl.train2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '2.aaai19srl.dev2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '2.aaai19srl.test2.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '3.aaai19srl.train3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '3.aaai19srl.dev3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '3.aaai19srl.test3.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    '4.aaai19srl.train4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '4.aaai19srl.dev4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    '4.aaai19srl.test4.conll.json': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'imdb-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'imdb-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'sst-2-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'sst-2-dev': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'sst-2-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'trec-train': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',
    'trec-test': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'all_zip': '<GOOGLE_DRIVE_URL_PLACEHOLDER>',

    'saved': SAVED_FILES
}

# Get direct download link
def get_download_url_from_google_drive_url(google_drive_url):
    return f'https://drive.google.com/uc?id={google_drive_url.split("/")[5]}&export=download&confirm=t'


In [ ]:
def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

In [ ]:
set_seed()

In [ ]:
FILE_PATHS = {"SST1": ("data/stsa.fine.phrases.train",
                  "data/stsa.fine.dev",
                  "data/stsa.fine.test"),
              "SST2": ("data/stsa.binary.phrases.train",
                  "data/stsa.binary.dev",
                  "data/stsa.binary.test"),
              "MR": ("data/rt-polarity.all", "", ""),
              "SUBJ": ("data/subj.all", "", ""),
              "CR": ("data/custrev.all", "", ""),
              "MPQA": ("data/mpqa.all", "", ""),
              "TREC": ("data/TREC.train.all", "", "data/TREC.test.all"),
              }

In [ ]:
def clean_str(string):
  """
  Tokenization/string cleaning for all datasets except for SST.
  """
  string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
  string = re.sub(r"\'s", " \'s", string)
  string = re.sub(r"\'ve", " \'ve", string)
  string = re.sub(r"n\'t", " n\'t", string)
  string = re.sub(r"\'re", " \'re", string)
  string = re.sub(r"\'d", " \'d", string)
  string = re.sub(r"\'ll", " \'ll", string)
  string = re.sub(r",", " , ", string)
  string = re.sub(r"!", " ! ", string)
  string = re.sub(r"\(", " ( ", string)
  string = re.sub(r"\)", " ) ", string)
  string = re.sub(r"\?", " ? ", string)
  string = re.sub(r"\s{2,}", " ", string)
  return string.strip().lower()

def clean_str_sst(string):
  """
  Tokenization/string cleaning for the SST dataset
  """
  string = re.sub(r"[^A-Za-z0-9(),!?\'\`]", " ", string)
  string = re.sub(r"\s{2,}", " ", string)
  return string.strip().lower()

In [ ]:
def line_to_words(line, dataset):
  if dataset == 'SST1' or dataset == 'SST2':
    clean_line = clean_str_sst(line.strip())
  else:
    clean_line = clean_str(line.strip())
  words = clean_line.split(' ')
  #words = words[1:]

  return words

In [ ]:
def load_data(dataset, train_name, test_name='', dev_name=''):
  """
  Load training data (dev/test optional).
  """

  # Initialize datasets
  train_dataset = dict({'input': [], 'output': []})
  dev_dataset = dict({'input': [], 'output': []})
  test_dataset = dict({'input': [], 'output': []})


  total_lines = list()

  f_train = open(train_name, 'r', encoding='ISO-8859-1')
  for line in f_train:
    words = line_to_words(line, dataset)
    input = str(' '.join(words[1:]))
    output = str(words[0])
    total_lines.append(line)
    train_dataset['input'].append(PREFIX[0] + input)
    train_dataset['output'].append(output)


  if not test_name == '':
    f_test = open(test_name, 'r', encoding='ISO-8859-1')
    for line in f_test:
      words = line_to_words(line, dataset)
      input = str(' '.join(words[1:]))
      output = str(words[0])
      total_lines.append(line)
      test_dataset[PREFIX[0] + input] = output
      test_dataset['input'].append(PREFIX[0] + input)
      test_dataset['output'].append(output)


  if not dev_name == '':
    f_dev = open(dev_name, 'r', encoding='ISO-8859-1')
    for line in f_dev:
      words = line_to_words(line, dataset)
      input = str(' '.join(words[1:]))
      output = str(words[0])
      total_lines.append(line)
      dev_dataset['input'].append(PREFIX[0] + input)
      dev_dataset['output'].append(output)


  print(len(total_lines))
  print(len(train_dataset['input']) + len(dev_dataset['input']) + len(test_dataset['input']))



  return train_dataset, dev_dataset, test_dataset, total_lines



In [ ]:
# Importing t5 models

from transformers import T5Tokenizer, T5ForConditionalGeneration, T5Model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [ ]:
# Instantiate tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
TOKENIZER[0] = tokenizer
# Adding our special tokens to the tokenizer
# tokenizer.add_tokens(['||', '|||', '{', '}', '<agent>', '<target>', '<exp>', '='])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
class SentimentDataset(Dataset):

  def __init__(self, texts, targets=None, max_len_in=64, max_len_out=32, custom=False):
    self.texts = texts
    self.targets = targets
    self.tokenizer = TOKENIZER[0]
    self.max_len_in = max_len_in
    self.max_len_out = max_len_out
    self.custom = custom

  def __len__(self):
    return len(self.texts)

  def __getitem__(self, item):
    text = self.texts[item]
    target = self.targets[item]

    encoding = self.tokenizer.batch_encode_plus(
            [text],
            max_length=self.max_len_in,
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    target_encoding = self.tokenizer.batch_encode_plus(
            [target],
            max_length=self.max_len_out,
            pad_to_max_length=True,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

    ######

    points = []
    if self.custom:
      len_finding_encoding = tokenizer.batch_encode_plus(
            [text],
            truncation=True,
            return_tensors="pt",
      )

      last_len = len(len_finding_encoding['input_ids'][0])
      ######
      real_ids = encoding['input_ids']
      real_att = encoding['attention_mask']
      #print(real_att)
      points = list(INDICES_OUT[0][COUNTER_SUBSET[0]])

      COUNTER_SUBSET[0] += 1

      if COUNTER_SUBSET[0] >= BOUND_SUBSET[0]:
        COUNTER_SUBSET[0] = 0

      backward = 0
      real_ids = real_ids[0, :]
      real_att = real_att[0, :]
      for ii in range(len(points)):
          points[ii] = points[ii] + backward
          point = points[ii]
          tt = torch.full((1, 1), 60000)[0, :]
          aa = torch.full((1, 1), 1)[0, :]
          real_ids = torch.cat([real_ids[0: point], tt, real_ids[point: ]], 0)
          real_att = torch.cat([real_att[0: point], aa, real_att[point: ]], 0)
          backward += 1
          #print(real_ids)
          #print(real_att)
          #print('Iteration finished!', '\n')
      real_ids = real_ids.clone().detach()
      real_att = real_att.clone().detach()
    else:
      real_ids = encoding['input_ids'].squeeze(),
      real_att = encoding['attention_mask'].squeeze(),
      real_ids = real_ids[0]
      real_att = real_att[0]

    return {
    'text': text,
    'source_ids': real_ids,
    'source_mask': real_att,
    'target_ids': target_encoding['input_ids'].float().squeeze(),
    'target_mask': target_encoding['attention_mask'].float().squeeze(),
    'max_len_in': self.max_len_in,
    'max_len_out': self.max_len_out,
    'points': torch.tensor(points)
    }

In [ ]:
def create_data_loader(texts, targets=None, max_len_in=64, max_len_out=32, batch_size=16, custom=False):
  ds = SentimentDataset(
    texts=texts,
    targets=targets,
    max_len_in=max_len_in,
    max_len_out=max_len_out,
    custom=custom
  )

  return DataLoader(
    ds,
    batch_size=batch_size,
    num_workers=0
  )

In [ ]:
def initilize_model(freeze_embedding=False):

    #model_hf = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    model_hf = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

    cnt = 0
    for n, p in model_hf.named_parameters():
        p.requires_grad = not False

    return model_hf


In [ ]:
import torch.optim as optim
import torch.nn as nn

def load_optimizer(model, learning_rate=1e-5, weight_decay=0):

    #optimizer = optim.AdamW(optimizer_grouped_parameters, lr=learning_rate)

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    return optimizer

In [ ]:
class SentimentClassifier(nn.Module):

  def __init__(self, t5model):
    super(SentimentClassifier, self).__init__()
    self.model = t5model

  def forward(self, ids, mask, y_ids=None, eval_time=False, max_len=64, points=None):

    if eval_time:
        outputs = self.model.generate(
                  input_ids = ids,
                  attention_mask = mask,
                  max_length=max_len,
            )

        # outputs = self.model.generate(
        #     input_ids = ids,
        #     attention_mask = mask,
        #     max_length=256,
        #     num_beams=NUM_BEAMS,
        #     repetition_penalty=REPITITION_PENALTY,
        #     length_penalty=LENGTH_PENALTY,
        #     early_stopping=True,
        #     do_sample=DO_SAMPLE
        #     )

    else:
        outputs = self.model(input_ids=ids,
                attention_mask=mask,
                labels=y_ids)

    return outputs


In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
def train_epoch(
  model,
  data_loaders,
  optimizer,
  device,
  n_examples,
  scheduler=None,
  epoch_num=1,
  eval_time=False,
  custom=False
):

  if eval_time:
    model = model.eval()
  else:
    model = model.train()


  losses = []
  correct_predictions = 0
  f1 = 0

  INDICATOR = [0, 0]

  for data_loader in data_loaders:
    for d in data_loader:

        points = d["points"]

        outputs = model(d["source_ids"].to(device, dtype=torch.long),
                        d["source_mask"].to(device, dtype=torch.long),
                        y_ids=d["target_ids"].to(device, dtype=torch.long)
                        )
        loss = outputs.loss

        INDICATOR[0] += BATCH_SIZE

        #loss = outputs[0]
        #loss, prediction_scores = outputs[:2]

        if not eval_time:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        losses.append(loss.item())



  return np.mean(losses)

In [ ]:
def eval_model(model, data_loaders, device, n_examples, generate=True):
  model = model.eval()

  losses = []
  correct_predictions = 0
  predictions = []
  actuals = []
  input_texts = []
  indices = []

  data_items = {}

  INDICATOR = [0, 0]

  for data_loader in data_loaders:
    with torch.no_grad():
      for d in data_loader:
        y = d['target_ids'].to(device, dtype = torch.long)
        ids = d['source_ids'].to(device, dtype = torch.long)
        mask = d['source_mask'].to(device, dtype = torch.long)
        input_text = d["text"]


        #inputs_dict = dict.fromkeys(input_text, [])

        if generate:

          generated_ids = model(
              ids,
              mask,
              eval_time=True,
              max_len=GENERATE_MAX_LEN)


          preds = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=True) for g in generated_ids]
          target = [tokenizer.decode(t, skip_special_tokens=True, clean_up_tokenization_spaces=True)for t in y]
          predictions.extend(preds)
          actuals.extend(target)
          input_texts.extend(input_text)

        else:
          points = d["points"]

          outputs = model(d["source_ids"].to(device, dtype=torch.long),
                          d["source_mask"].to(device, dtype=torch.long),
                          y_ids=d["target_ids"].to(device, dtype=torch.long),
                          eval_time=False,
                          points=points
                          )
          loss = outputs.loss
          losses.append(loss.item())

        INDICATOR[0] += 1



        #

  if generate:
    return predictions, actuals, input_texts
  else:
    return np.mean(losses)

In [ ]:
def clear_string(txt):
  txt = txt.lower()

  if txt.startswith('{'):
    txt = txt[1:]

  if txt.endswith('}'):
    txt = txt[:-1]

  txt = re.sub(':', ' ', txt)
  txt = re.sub('{', ' ', txt)
  txt = re.sub('}', ' ', txt)
  txt = re.sub(' = ', ' ', txt)
  txt = re.sub('=', ' ', txt)
  txt = re.sub('  ', ' ', txt)
  txt = re.sub(' ', '', txt)
  txt = re.sub('^ ', '', txt)
  txt = re.sub('|', '', txt)
  txt = re.sub(' $', '', txt)
  txt = re.sub('|$', '', txt)
  txt = re.sub('^|', '', txt)
  txt = re.sub('^:', '', txt)
  txt = re.sub(':$', '', txt)
  return txt

In [ ]:
def calculate_metrics(actuals, predictions):
  results = {'acc': 0, 'f1_micro': 0, 'f1_macro': 0, 'f1_weighted': 0}

  if EVALUATION_METHOD == 'SIMPLE':
    correct = 0
    for i in range(len(actuals)):
      if actuals[i] == predictions[i]:
        correct += 1
    results['acc'] = correct / len(actuals)
    # average can be 'micro', 'macro', 'weighted', or 'samples'
    for f1_type in ['micro', 'macro', 'weighted']:
      results[f'f1_{f1_type}'] = f1_score(actuals, predictions, average=f1_type)

  elif EVALUATION_METHOD == 'MULTIPLE':
    correct = 0
    all_predictions = 0
    all_actuals = 0
    for i in range(len(actuals)):
      actual_sep = actuals[i].split('|')
      all_actuals += len(actual_sep)
      prediction_sep = predictions[i].split('|')
      all_predictions += len(prediction_sep)
      new_prediction_sep  = [clear_string(x) for x in prediction_sep]
      for actual_item in actual_sep:
        cleared_actual_item = clear_string(actual_item)
        if cleared_actual_item in new_prediction_sep:
          correct += 1

    precision = correct / all_predictions
    recall = correct / all_actuals
    results['f1'] = (2 * precision * recall) / (precision + recall)
    # print(actuals[ : 4])
    # print(predictions[ : 4])

  return results

In [ ]:
def train_eval(model, train_data_loaders, val_data_loaders, test_data_loaders,
               custom_train_data_loaders,
               train_set_size, val_set_size, test_set_size,
               custom_train_data_loaders_size,
               fold, epochs=4):

  history = defaultdict(list)
  best_performance_dev = {'acc': 0, 'f1_micro': 0, 'f1_macro': 0, 'f1_weighted': 0, 'loss': float('inf')}
  best_performance_test = {'acc': 0, 'f1_micro': 0, 'f1_macro': 0, 'f1_weighted': 0, 'loss': float('inf')}

  # Start training loop
  print("Start training...\n")

  for epoch in range(epochs):

    print(f'Epoch {epoch + 1}/{epochs}')
    print('-' * 20)
    start_time = time.time()

    loss = train_epoch(
      model,
      train_data_loaders,
      optimizer,
      device,
      train_set_size,
      epoch_num=epoch
    )

    print(f'Train loss {loss}')

    if ENABLE_AUG[0]:
      AUG_STATUS[0] = True
      aug_loss = train_epoch(
        model,
        custom_train_data_loaders,
        optimizer,
        device,
        custom_train_data_loaders_size,
        epoch_num=epoch,
        custom=True
      )
      AUG_STATUS[0] = False
      INDICATOR[0] = 0

      print(f'Aug Train loss {aug_loss}')

    #

    if EVALUATE[0]:


      if GENERATE[0]:
        if True:
          loss = eval_model(
            model,
            val_data_loaders,
            device,
            val_set_size,
            generate=False
          )
          print(f'It just came to the market ({loss}).')
          if loss < best_performance_dev['loss']:
            old_loss = best_performance_dev['loss']
            print(f'It just came to the market ({loss}), it got old ({old_loss}), it hurts.')
            best_performance_dev['loss'] = loss
            predictions, actuals, input_texts = eval_model(
              model,
              test_data_loaders,
              device,
              test_set_size,
              generate=True
            )
            test_results = calculate_metrics(actuals, predictions)
            print('Evaluation result of Test set: ', test_results)
            for test_results_key in test_results.keys():
              best_performance_test[test_results_key] = test_results[test_results_key]



        else:
          loss = eval_model(
            model,
            test_data_loaders,
            device,
            test_set_size,
            generate=False
          )

          if loss < best_performance_test['loss']:
            best_performance_test['loss'] = loss
            predictions, actuals, input_texts = eval_model(
              model,
              test_data_loaders,
              device,
              test_set_size,
              generate=True
            )
            test_results = calculate_metrics(actuals, predictions)
            print('Evaluation result of Test set: ', test_results)
            for test_results_key in test_results.keys():
              best_performance_test[test_results_key] = test_results[test_results_key]

          # print('\npred::\n')
          # print(predictions[ : 20])

          # print('\nacts::\n')
          # print(actuals[ : 20])
      else:
        loss = eval_model(
          model,
          test_data_loaders,
          device,
          test_set_size,
          generate=False
        )
        print(f'Test loss {loss}')




    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch Time: {epoch_mins}m {epoch_secs}s\n')

  return best_performance_test

In [ ]:
def make_custom_dataloader(train, y_train):
  max_len_cnt_train = [256]
  token_len_cnt_train = [0] * len(max_len_cnt_train)
  out_max_len_cnt_train = [0] * len(max_len_cnt_train)
  trains = []
  y_trains = []

  for i in range(len(max_len_cnt_train)):
    trains.append(list())
    y_trains.append(list())


  token_lens = []

  for k in range(len(train)):

    tokens1 = tokenizer(train[k])['input_ids']
    tokens2 = tokenizer(y_train[k])['input_ids']
    token_lens.append(max(len(tokens1), len(tokens2)))

    for i in range(len(max_len_cnt_train)):
      if max(len(tokens1), len(tokens2)) <= max_len_cnt_train[i]:
        token_len_cnt_train[i] += 1
        trains[i].append(train[k])
        y_trains[i].append(y_train[k])
        out_max_len_cnt_train[i] = max(out_max_len_cnt_train[i], len(tokens2))
        break

  #new method
  train_data_loaders = []

  for i in range(len(trains)):
    train_data_loader = create_data_loader(trains[i], targets=y_trains[i], max_len_in=max_len_cnt_train[i], max_len_out=out_max_len_cnt_train[i], batch_size=AUG_BATCH_SIZE, custom=True)
    train_data_loaders.append(train_data_loader)

  return train_data_loaders

In [ ]:
def make_dataloaders(train, val, test, y_train, y_val, y_test, indices_val=None, indices_test=None):
  max_len_cnt_train = [MAX_LENGTHS_ARRAY[DATASET_NAME] + STUFF_COUNT]
  token_len_cnt_train = [0] * len(max_len_cnt_train)
  out_max_len_cnt_train = [0] * len(max_len_cnt_train)
  trains = []
  y_trains = []

  for i in range(len(max_len_cnt_train)):
    trains.append(list())
    y_trains.append(list())


  token_lens = []

  for k in range(len(train)):

    tokens1 = tokenizer(train[k])['input_ids']
    tokens2 = tokenizer(y_train[k])['input_ids']
    token_lens.append(max(len(tokens1), len(tokens2)))

    for i in range(len(max_len_cnt_train)):
      if max(len(tokens1), len(tokens2)) <= max_len_cnt_train[i]:
        token_len_cnt_train[i] += 1
        trains[i].append(train[k])
        y_trains[i].append(y_train[k])
        out_max_len_cnt_train[i] = max(out_max_len_cnt_train[i], len(tokens2))
        break


  max_len_cnt_val = max_len_cnt_train
  token_len_cnt_val = [0] * len(max_len_cnt_val)
  out_max_len_cnt_val = [0] * len(max_len_cnt_val)
  vals = []
  y_vals = []

  for i in range(len(max_len_cnt_val)):
    vals.append(list())
    y_vals.append(list())

  for k in range(len(val)):
    tokens1 = tokenizer(val[k])['input_ids']
    tokens2 = tokenizer(y_val[k])['input_ids']
    token_lens.append(max(len(tokens1), len(tokens2)))

    for i in range(len(max_len_cnt_val)):
      if max(len(tokens1), len(tokens2)) <= max_len_cnt_val[i]:
        token_len_cnt_val[i] += 1
        vals[i].append(val[k])
        y_vals[i].append(y_val[k])
        out_max_len_cnt_val[i] = max(out_max_len_cnt_val[i], len(tokens2))
        break

  max_len_cnt_test = max_len_cnt_val
  token_len_cnt_test = [0] * len(max_len_cnt_test)
  out_max_len_cnt_test = [0] * len(max_len_cnt_test)
  tests = []
  y_tests = []
  for i in range(len(max_len_cnt_test)):
    tests.append(list())
    y_tests.append(list())

  for k in range(len(test)):
    tokens1 = tokenizer(test[k])['input_ids']
    tokens2 = tokenizer(y_test[k])['input_ids']
    token_lens.append(max(len(tokens1), len(tokens2)))

    for i in range(len(max_len_cnt_test)):
      if max(len(tokens1), len(tokens2)) <= max_len_cnt_test[i]:
        token_len_cnt_test[i] += 1
        tests[i].append(test[k])
        y_tests[i].append(y_test[k])
        out_max_len_cnt_test[i] = max(out_max_len_cnt_test[i], len(tokens2))
        break

  #new method
  train_data_loaders = []
  val_data_loaders = []
  test_data_loaders = []

  for i in range(len(trains)):
    train_data_loader = create_data_loader(trains[i], targets=y_trains[i], max_len_in=max_len_cnt_train[i], max_len_out=out_max_len_cnt_train[i], batch_size=TRAIN_BATCH_SIZE)
    train_data_loaders.append(train_data_loader)


  for i in range(len(vals)):
    val_data_loader = create_data_loader(vals[i], targets=y_vals[i], max_len_in=max_len_cnt_val[i], max_len_out=out_max_len_cnt_val[i], batch_size=VAL_BATCH_SIZE)
    val_data_loaders.append(val_data_loader)


  for i in range(len(tests)):
    test_data_loader = create_data_loader(tests[i], targets=y_tests[i], max_len_in=max_len_cnt_test[i], max_len_out=out_max_len_cnt_test[i], batch_size=TEST_BATCH_SIZE)
    test_data_loaders.append(test_data_loader)



  return train_data_loaders, val_data_loaders, test_data_loaders

# Dataset Creation

In [ ]:
class SoftEmbedding(nn.Module):
    def __init__(self,
                wte: nn.Embedding,
                n_tokens: int = 10,
                random_range: float = 0.5,
                initialize_from_vocab: bool = True):

        super(SoftEmbedding, self).__init__()
        self.wte = wte
        self.n_tokens = n_tokens

        params_list = []
        for i in range(BOUND_SUBSET[0]):
          for j in range(STUFF_COUNT):
            params_list.append(nn.parameter.Parameter(self.initialize_embedding(wte,
                                                                                1,
                                                                                random_range,
                                                                                initialize_from_vocab)))

        self.mid_learned_embedding = nn.ParameterList(params_list)


    def initialize_embedding(self,
                             wte: nn.Embedding,
                             n_tokens: int = 10,
                             random_range: float = 0.5,
                             initialize_from_vocab: bool = True):

        tt = torch.FloatTensor(n_tokens, wte.weight.size(1))
        sam = random.sample(range(0, wte.weight.size(0)), n_tokens)
        tt = self.wte.weight[sam].clone().detach()

        return tt

    def forward(self, tokens):

        #sentence_tokens = tokens[:, self.n_tokens:]
        sentence_tokens = tokens[:, :]
        if AUG_STATUS[0]:
          sentence_tokens = tokens[:, :]
          sentences_embs = torch.full((tokens.shape[0], tokens.shape[1], self.wte.weight.size(1)), 0, dtype = torch.float).to(device)
          indices = [None] * sentence_tokens.shape[0]
          for i in range(sentence_tokens.shape[0]):
            backward = 0
            for j in range(sentence_tokens[i, :].shape[0]):
              if sentence_tokens[i, j] == 60000:
                if indices[i]:
                  indices[i].append(j + backward)
                else:
                  indices[i] = [j + backward]
                #backward += 1

            # The last of us
            if indices[i]:
              indices[i].append(sentence_tokens[i, :].shape[0] + backward)
            else:
              indices[i] = [sentence_tokens[i, :].shape[0] + backward]

          segments = []

          for i in range(tokens.shape[0]):
            # For the general case
            # STUFF_COUNT gets involved
            input_embedding = []
            for k in range(STUFF_COUNT):
              input_embedding.append(self.mid_learned_embedding[STUFF_COUNT * i + k].to(device))

            # Now we should paste the parts together
            progress_list = [self.wte(sentence_tokens[i, 0: indices[i][0]])]

            for k in range(STUFF_COUNT):
              progress_list.append(input_embedding[k])
              current_segment = self.wte(sentence_tokens[i, indices[i][k] + 1: indices[i][k + 1]])
              progress_list.append(current_segment)

            # Build the tensor and put it in the right place
            final = torch.cat(progress_list, 0)
            sentences_embs[i, :, :] = final

          INDICATOR[0] += AUG_BATCH_SIZE

        else:
            sentences_embs = self.wte(sentence_tokens[:, :])
        return sentences_embs

In [ ]:
def create_piped(sentence, annots):
    span = ''
    the_sentence = detokenizer.detokenize(sentence)
    the_span = ''
    the_target = ''
    the_agent = ''
    itself = {}
    span = ''
    agent = ''
    target = ''
    role_target = ''
    role_agent = ''
    role_exp = ''

    for item1 in annots:
        if item1[4] == 'DSE':
            sentence_prime = sentence.copy()
            the_agent = ' '
            the_target = ' '


            for item2 in annots:
                if item2[0] == item1[0] and item2[1] == item1[1]:
                    if item2[4] == 'TARGET':
                        the_target = detokenizer.detokenize(sentence[item2[2]: item2[3] + 1])

                    elif item2[4] == 'AGENT':
                        the_agent = detokenizer.detokenize(sentence[item2[2]: item2[3] + 1])




            the_span = detokenizer.detokenize(sentence[item1[0]: item1[1] + 1])

            role_exp = role_exp + ' | ' + the_span

    if role_exp.startswith(' | '):
        role_exp = role_exp[3:]

    if span.startswith(' | '):
        span = span[3:]


    role_exp = '{ ' + role_exp + ' }'

    sample = {'role_exp': role_exp}

    return sample

In [ ]:
def mpqa_tpi_data_reader(data_name):
  if data_name == 'MPQA-T':
      trainset = pd_read_csv(mpqa_t_train_link)
      valset   = pd_read_csv(mpqa_t_val_link)
      testset  = pd_read_csv(mpqa_t_test_link)
  elif data_name == 'MPQA-P':
      trainset = pd_read_csv(mpqa_p_train_link)
      valset   = pd_read_csv(mpqa_p_val_link)
      testset  = pd_read_csv(mpqa_p_test_link)
  elif data_name == 'MPQA-I':
      trainset = pd_read_csv(mpqa_i_train_link)
      valset   = pd_read_csv(mpqa_i_val_link)
      testset  = pd_read_csv(mpqa_i_test_link)

  X_train, y_train = decompose_mpqa(trainset)
  X_val,   y_val   = decompose_mpqa(valset)
  X_test,  y_test  = decompose_mpqa(testset)

  # Extract data samples
  train_dataset = dict()
  val_dataset = dict()
  test_dataset = dict()

  for i in range(len(X_train)):
      train_dataset[PREFIX[0] + X_train[i]] = y_train[i]

  for i in range(len(X_val)):
      val_dataset[PREFIX[0] + X_val[i]] = y_val[i]

  for i in range(len(X_test)):
      test_dataset[PREFIX[0] + X_test[i]] = y_test[i]

  return train_dataset, val_dataset, test_dataset

In [ ]:
def sentiment_tsv_data_reader(data_name, conversion_dict, col_map):
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  response = urlopen(data_url)
  # Get path to file in Colab
  file_path = io.BytesIO(response.read())
  # Read TSV file into DataFrame
  df = pd.read_csv(file_path, sep='\t', header=0, on_bad_lines='skip')
  # Extract data samples
  dataset = dict()
  print(df.columns)

  for iterator in range(df.shape[0]):

    input = df.at[iterator, col_map['input']]
    output = df.at[iterator, col_map['output']]
    dataset[PREFIX[0] + input] = conversion_dict[output]

  return dataset


In [ ]:
def mpqa_data_reader(data_name):
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  response = urlopen(data_url)
  dd = response.readlines()
  data = list()
  # Extract data samples
  dataset = dict()


  counter = 0

  for line in dd:
      counter += 1
      if counter > len(dd):
          break

      data.append(json.loads(line.decode()))

  #
  for item in data:
    sentence = item['sentences']
    annots = item['orl']
    sentence_output = create_piped(sentence, annots)

    dataset[PREFIX[0] + detokenizer.detokenize(sentence)] = sentence_output['role_exp']

  return dataset

In [ ]:
def trec_data_reader(data_name, conversion_dict=None, col_map=None):
  trec_dataset = load_dataset('trec')
  train_dataset = trec_dataset[data_name]
  dataset = dict()
  for item in train_dataset:
    input = item['text']
    output = item['coarse_label']
    dataset[PREFIX[0] + input] = output if not conversion_dict else conversion_dict[output]

  # google_drive_url = data_name_to_google_drive_url[data_name]
  # data_url = get_download_url_from_google_drive_url(google_drive_url)
  # response = urlopen(data_url)
  # df = pd.read_json(response)

  # print(df.columns)
  # print(df.shape)
  # unique_values = df['label'].unique()
  # print(unique_values)

  # # Extract data samples
  # dataset = dict()

  # for iterator in range(df.shape[0]):

  #   input = df.at[iterator, col_map['input']]
  #   output = df.at[iterator, col_map['output']]
  #   dataset[PREFIX[0] + input] = output if not conversion_dict else conversion_dict[output]


  return dataset

In [ ]:
def clean_str_sst(string):
    return string

def clean_str(string):
    return string

def line_to_words(line, dataset):
    if dataset == 'SST1' or dataset == 'SST2':
        clean_line = clean_str_sst(line.strip())
    else:
        clean_line = clean_str(line.strip())
    words = line.split(' ')
    return words

def load_data(dataset_name, train_name, test_name='', dev_name=''):
    """
    Load training data (dev/test optional).
    If test set is not provided, use 20% of the train set as the test set.
    """
    dataset = {
        'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},
        'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}
    }

    total_lines = []

    # Set the random seed to ensure reproducibility
    #random.seed(42)

    # Load training data
    with open(train_name, 'r', encoding='ISO-8859-1') as f_train:
        for line in f_train:
            words = line_to_words(line, dataset_name)
            input = ' '.join(words[1:])
            output = words[0]
            total_lines.append((input, output))

    # If no test set is provided, use 20% of the training data as the test set
    if test_name == '':
        random.shuffle(total_lines)
        split_index = int(len(total_lines) * 0.8)
        train_lines = total_lines[:split_index]
        test_lines = total_lines[split_index:]
    else:
        train_lines = total_lines
        test_lines = []
        with open(test_name, 'r', encoding='ISO-8859-1') as f_test:
            for line in f_test:
                words = line_to_words(line, dataset_name)
                input = ' '.join(words[1:])
                output = words[0]
                test_lines.append((input, output))

    # Populate train dataset
    for input, output in train_lines:
        dataset['train']['text'].append(input)
        dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]].append(
            LABEL_CONVERTOR[DATASET_NAME][output]
        )

    # Populate test dataset
    for input, output in test_lines:
        dataset['test']['text'].append(input)
        dataset['test'][LABEL_KEY_MAPPER[DATASET_NAME]].append(
            LABEL_CONVERTOR[DATASET_NAME][output]
        )

    # Load dev data if provided
    if dev_name:
        with open(dev_name, 'r', encoding='ISO-8859-1') as f_dev:
            for line in f_dev:
                words = line_to_words(line, dataset_name)
                input = ' '.join(words[1:])
                output = words[0]
                dataset['train']['text'].append(input)
                dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]].append(
                    LABEL_CONVERTOR[DATASET_NAME][output]
                )

    print(f"Train lines: {len(train_lines)}")
    print(f"Test lines: {len(test_lines)}")

    return dataset


In [ ]:
import requests

def fetch_and_process_r8_data(url):
    """
    Downloads and processes the R8 dataset from the specified URL.

    Args:
        url (str): The URL to the raw train.txt file.

    Returns:
        tuple: Two lists - one with input texts and another with corresponding labels.
    """
    try:
        # Send a GET request to the URL
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors

        # Decode the content to a string
        content = response.content.decode('utf-8')

        # Initialize lists to hold inputs and outputs
        inputs = []
        outputs = []

        # Process each line in the content
        for line in content.splitlines():
            # Split the line into label and text
            parts = line.split(maxsplit=1)
            if len(parts) == 2:
                label, text = parts
                try:
                  outputs.append(LABEL_CONVERTOR[DATASET_NAME][label])
                  inputs.append(text)
                except:
                  print('Error on line.')
            else:
                print(f"Skipping malformed line: {line}")

        return inputs, outputs

    except requests.exceptions.RequestException as e:
        print(f'Error downloading the file: {e}')
        return [], []

In [ ]:
def convert_to_t5_input(input):
  mpqa_type, mpqa_exp, mpqa_sentence = input.split('[SEP]')
  mpqa_type = mpqa_type.strip()
  mpqa_exp = mpqa_exp.strip()
  mpqa_sentence = mpqa_sentence.strip()
  new_input = "type: " + mpqa_type + ". expression: " + mpqa_exp + ". sentence: " + mpqa_sentence + "."
  return new_input

In [ ]:
def get_pte_datasets(dataset_name=DATASET_NAME):
  dataset_name = dataset_name.lower()
  dataset = {'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},\
              'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}}
  # URLs for the raw files
  urls = {
      "text_train": f"https://raw.githubusercontent.com/mnqu/PTE/master/data/{dataset_name}/text_train.txt",
      "label_train": f"https://raw.githubusercontent.com/mnqu/PTE/master/data/{dataset_name}/label_train.txt",
      "text_test": f"https://raw.githubusercontent.com/mnqu/PTE/master/data/{dataset_name}/text_test.txt",
      "label_test": f"https://raw.githubusercontent.com/mnqu/PTE/master/data/{dataset_name}/label_test.txt"
  }

  # Function to download file contents
  def download_file(url):
      response = requests.get(url)
      response.raise_for_status()  # Check if the request was successful
      return response.text.splitlines()  # Return as list of lines

  # Download the files
  text_train = download_file(urls["text_train"])
  label_train = download_file(urls["label_train"])
  text_test = download_file(urls["text_test"])
  label_test = download_file(urls["label_test"])

  # Convert labels to integers
  label_train = [int(label) for label in label_train]
  label_test = [int(label) for label in label_test]

  # Combine texts with labels to form train and test sets
  train_data = list(zip(text_train, label_train))
  test_data = list(zip(text_test, label_test))

  # Separate texts and labels for easy access
  train_texts, train_labels = zip(*train_data)
  test_texts, test_labels = zip(*test_data)

  # Optionally, convert to list format for flexibility
  dataset['train']['text'], dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]] = list(train_texts), list(train_labels)
  dataset['test']['text'], dataset['test'][LABEL_KEY_MAPPER[DATASET_NAME]] = list(test_texts), list(test_labels)

  return dataset

In [ ]:
import requests

def download_file_r52_trec(url):
    """
    Downloads file from the given URL.
    """
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def parse_corpus_r52_trec(corpus_text):
    """
    Parses the corpus text into a list where each element is a document.
    """
    return corpus_text.strip().split('\n')

def parse_items_r52_trec(items_text):
    """
    Parses the items information and returns a list of tuples with (index, set, class label).
    """
    items = []
    for line in items_text.strip().split('\n'):
        parts = line.split()
        if len(parts) == 3:
            index, set_type, label = parts
            items.append((int(index), set_type, label))
    return items

def create_datasets_r52_trec(corpus, items):
    """
    Create train and test datasets based on items information.
    """
    dataset = {'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},\
                'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}}
    train_data = []
    test_data = []

    for index, set_type, label in items:
        document = corpus[index]  # Retrieve document using the index
        if set_type == 'train':
          dataset['train']['text'].append(document)
          dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]].append(LABEL_CONVERTOR[DATASET_NAME][label])
        elif set_type == 'test':
          dataset['test']['text'].append(document)
          dataset['test'][LABEL_KEY_MAPPER[DATASET_NAME]].append(LABEL_CONVERTOR[DATASET_NAME][label])

    return dataset

def fetch_and_process_r52_trec_data():
    # URLs
    if DATASET_NAME == "trec":
      corpus_url = 'https://raw.githubusercontent.com/FKarl/short-text-classification/main/data/corpus/TREC.txt'
      items_url = 'https://raw.githubusercontent.com/FKarl/short-text-classification/main/data/TREC/TREC.txt'
    else:
      corpus_url = 'https://raw.githubusercontent.com/ZeroRin/BertGCN/refs/heads/main/data/corpus/R52.txt'
      items_url = 'https://raw.githubusercontent.com/ZeroRin/BertGCN/main/data/R52.txt'

    # Download the files
    corpus_text = download_file_r52_trec(corpus_url)
    items_text = download_file_r52_trec(items_url)

    # Parse the content
    corpus = parse_corpus_r52_trec(corpus_text)
    items = parse_items_r52_trec(items_text)

    # Create dataset
    dataset = create_datasets_r52_trec(corpus, items)

    return dataset


In [ ]:

folds_performance_test = {'acc': 0, 'f1_micro': 0, 'f1_macro': 0, 'f1_weighted': 0, 'loss': float('inf')}
#
ONCE_DONE = [False, False]
ACTUAL_TRIPLETS = [{}, {}]
GOLD_NUM = [None, None]

# Initialize datasets
train_dataset = dict({'input': [], 'output': []})
dev_dataset = dict({'input': [], 'output': []})
test_dataset = dict({'input': [], 'output': []})


if DATASET_NAME in ["R8"]:
  dataset = {'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},\
              'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}}
  # URL of the raw train.txt file in the GitHub repository
  url = 'https://raw.githubusercontent.com/FKarl/short-text-classification/main/data/R8/train.txt'

  # Fetch and process the data
  dataset['train']['text'], dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]] = fetch_and_process_r8_data(url)

  url = 'https://raw.githubusercontent.com/FKarl/short-text-classification/main/data/R8/test.txt'

  # Fetch and process the data
  dataset['test']['text'], dataset['test'][LABEL_KEY_MAPPER[DATASET_NAME]] = fetch_and_process_r8_data(url)

elif DATASET_NAME in ['laptop14', 'rest14', 'rest15', 'rest16']:
  dataset = read_process_absa()

elif DATASET_NAME in ["ohsumed"]:
  #
  import requests
  # URL of the .tar.gz file
  url = "http://disi.unitn.it/moschitti/corpora/ohsumed-all-docs.tar.gz"

  # Local filename to save the downloaded file
  local_filename = "ohsumed-all-docs.tar.gz"

  # Download the file
  response = requests.get(url, stream=True)
  if response.status_code == 200:
      with open(local_filename, "wb") as file:
          for chunk in response.iter_content(chunk_size=8192):
              file.write(chunk)
      print(f"Downloaded '{local_filename}' successfully.")
  else:
      print(f"Failed to download file. Status code: {response.status_code}")


  ##
  import tarfile

  # The path to the .tar.gz file
  file_path = "ohsumed-all-docs.tar.gz"

  # The directory where you want to extract the contents
  extract_path = "ohsumed_docs"

  # Open the tar.gz file
  with tarfile.open(file_path, "r:gz") as tar:
      tar.extractall(path=extract_path)
      print(f"Extracted contents to '{extract_path}'")


  ##
  %cd ohsumed_docs/ohsumed-all

  import requests

  # URL to the raw text file in the GitHub repository
  url = "https://raw.githubusercontent.com/ZeroRin/BertGCN/main/data/ohsumed.txt"

  # Download the file content
  response = requests.get(url)
  if response.status_code == 200:
      data = response.text
  else:
      raise Exception(f"Failed to download file: Status code {response.status_code}")

  # Split file into rows
  rows = data.strip().split('\n')

  # Initialize lists for training and testing sets
  train_set = []
  test_set = []
  document_ids = []

  dataset = {'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},\
              'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}}

  # Parse each row
  for row in rows:
      parts = row.split('\t')

      if len(parts) < 3:  # Ensure there are enough columns
          continue

      document_id = int(parts[0].split('/')[-1]) # Path to the document
      document_type = parts[1]    # Document ID (could be used for reference)
      category = parts[2]        # Category label (e.g., C01)

      file_path = '/'.join(parts[0].split('/')[-2:])

      # Open and read a text file
      with open(file_path, "r", encoding="utf-8") as file:
          content = file.read()

      try:
        # Check if it's a test or train document based on the path
        if document_type.find('test') >= 0:
          dataset['test']['text'].append(content)
          dataset['test'][LABEL_KEY_MAPPER[DATASET_NAME]].append(LABEL_CONVERTOR[DATASET_NAME][category])
        elif document_type.find('train') >= 0:
          dataset['train']['text'].append(content)
          dataset['train'][LABEL_KEY_MAPPER[DATASET_NAME]].append(LABEL_CONVERTOR[DATASET_NAME][category])
      except:
        print('Error!')
        print(parts)

elif DATASET_NAME in ["SENTIPERS"]:
  dataset = read_sentipers()

elif DATASET_NAME in ["MR", "20NG", "dblp"]:
  dataset = get_pte_datasets()

elif DATASET_NAME in ["R52"]:
  dataset = fetch_and_process_r52_trec_data()

elif DATASET_NAME in ["GoEmotions"]:
  # Load dataset and prepare the custom structure
  ds = load_dataset("google-research-datasets/go_emotions", "simplified")
  dataset = {
      'train': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},
      'validation': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []},
      'test': {'text': [], LABEL_KEY_MAPPER[DATASET_NAME]: []}
  }

  # Loop through each subset and filter directly
  for subset in ['train', 'test', 'validation']:
      subset_data = ds[subset]
      filtered_data = [
          (text, label[0])
          for text, label in zip(subset_data['text'], subset_data['labels'])
          if len(label) == 1 and label[0] < 27
      ]
      dataset[subset]['text'], dataset[subset][LABEL_KEY_MAPPER[DATASET_NAME]] = zip(*filtered_data)

elif DATASET_NAME in ["CR"]:
  data_name = 'all_zip'
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  urllib.request.urlretrieve(data_url, "response.zip")
  os.system('unzip response.zip')
  train_path, dev_path, test_path = FILE_PATHS[DATASET_NAME]
  dataset = load_data(DATASET_NAME, train_path, test_name=test_path, dev_name=dev_path)
else:
  # Load dataset
  from datasets import load_dataset

  dataset = load_dataset(DATASET_NAME)

else:
  data_name = 'all_zip'
  google_drive_url = data_name_to_google_drive_url[data_name]
  data_url = get_download_url_from_google_drive_url(google_drive_url)
  #urllib.request.urlretrieve(data_url, "response.zip")
  os.system('unzip response.zip')
  train_path, dev_path, test_path = FILE_PATHS[DATASET_NAME]
  train_dataset, dev_dataset, test_dataset, total_lines = load_data(DATASET_NAME, train_path, test_name=test_path, dev_name=dev_path)



# Set seed
set_seed()

#
# Create a list of indices and shuffle it
indices = list(range(len(train_dataset['input'])))
random.shuffle(indices)

# Use the shuffled indices to rearrange your lists
train_dataset['input'] = [train_dataset['input'][i] for i in indices]
train_dataset['output'] = [train_dataset['output'][i] for i in indices]
#

folds_range = range(1, FOLDS + 1)

predefined_seed_set = [0, 1, 2]
predefined_seed_set = [76, 0, 1, 2]

# Training folds.
for fold in folds_range:
    step_folder = -1
    for ff in range(len(STEP_FOLDS)):
      if fold in STEP_FOLDS[ff]:
        step_folder = ff

    start_time = time.time()  # Start timer
    best_performance_test = {'acc': 0, 'f1_micro': 0, 'f1_macro': 0, 'f1_weighted': 0, 'loss': float('inf')}

    train = train_dataset['input']
    train_label = train_dataset['output']
    test = test_dataset['input']
    test_label = test_dataset['output']

    INIT_W = [None]
    BEST_W = [None]
    BEST_LOSS = [float('inf')]

    INDICATOR = [0, 0]

    INDICES_OUT = [list(), BOUND_SUBSET[0]]
    print()
    print(f'==> fold {fold}')

    if DATASET_NAME not in HAS_TEST:
        # make train/test data (90/10 split for train/test)
        N = len(train)
        i_start = int((fold - 1) * (N / FOLDS))
        i_end = int(fold * (N / FOLDS))

        test = train[i_start:i_end]
        test_label = train_label[i_start:i_end]

        train = train[:i_start] + train[i_end:]
        train_label = train_label[:i_start] + train_label[i_end:]


    # shuffle train to get dev/train split (10% to dev)
    J = len(train)

    # Generate a shuffled array of indices from 0 to J-1
    shuffle = np.random.permutation(J)

    # Use the shuffled indices to reorder the train data and labels
    # Create an index list from 0 to J-1
    indexes = list(range(J))


    # Shuffle the index list
    random.shuffle(indexes)

    # Use the shuffled indexes to reorder the train data and labels using list comprehension
    train = [train[i] for i in indexes]
    train_label = [train_label[i] for i in indexes]


    num_batches = J // BATCH_SIZE

    num_train_batches = round(num_batches * 0.9)
    print(type(num_train_batches))

    train_size = num_train_batches * BATCH_SIZE

    dev_size = J - train_size

    dev = train[train_size:]
    dev_label = train_label[train_size:]

    train = train[:train_size]
    train_label = train_label[:train_size]

    model_hf = initilize_model(freeze_embedding=False)

    s_wte = SoftEmbedding(model_hf.get_input_embeddings(),
                          initialize_from_vocab=True)


    # Load train aug data here and convert JSON to two lists just like the previous ones
    # Note: We could import data from multiple sources (i.e. produced with different seeds)
    ##
    # Create lists of data
    X_train_text_custom = list()
    y_train_custom = list()
    # A set to control the stuff count to be equal
    STUFF_COUNT_SET = set()
    source_count = 0
    SEED_SET = set()
    list_of_tensors = list()
    # Iterate over different origins of data
    for google_drive_url in data_name_to_google_drive_url[SAVED_ITEM_KEY]:
      source_count += 1
      if source_count > MAX_SOURCES_COUNT:
        break

      #os.system('rm -rf *')
      google_drive_url_folded = google_drive_url[step_folder]
      data_url = get_download_url_from_google_drive_url(google_drive_url_folded)

      #urllib.request.urlretrieve(data_url, f"response_data_{predefined_seed_set[source_count - 1]}.zip")
      # os.system(f'unzip response_data_{predefined_seed_set[source_count - 1]}.zip')

      urllib.request.urlretrieve(data_url, "response_data.tar.gz")
      os.system('tar -xzf response_data.tar.gz')
      print(data_url)
      f = open(f'aug-train-{fold}-{predefined_seed_set[source_count - 1]}.json')

      json_data = json.load(f)
      f.close()
      #os.system(f'rm -rf aug-train-{fold}.json')
      X_train_text_custom.extend(json_data['dataset']['input'])
      y_train_custom.extend(json_data['dataset']['output'])
      INDICES_OUT[0].extend(json_data['indices']) #INDICES_OUT[0] = json_data['indices']
      AUG_BATCH_SIZE = json_data['batch_size']
      COUNTER_SUBSET[0] = 0
      #BOUND_SUBSET[0] = len(INDICES_OUT[0])
      SEED_SET.add(json_data['seed'])
      STUFF_COUNT = json_data['stuff_count']
      STUFF_COUNT_SET.add(STUFF_COUNT)
      print(f'Fetched data from source {source_count}.')
      if len(list_of_tensors) == 0:
        print(type(torch.load(f'mid_learned_embedding_{fold}_{predefined_seed_set[source_count - 1]}.pth')))
        list_of_tensors = [torch.load(f'mid_learned_embedding_{fold}_{predefined_seed_set[source_count - 1]}.pth')]
      else:
        list_of_tensors.extend(torch.load(f'mid_learned_embedding_{fold}_{predefined_seed_set[source_count - 1]}.pth'))
      print(f'Fetched weights from source {source_count}.')

    # Check
    assert len(STUFF_COUNT_SET) == 1
    BOUND_SUBSET[0] = len(INDICES_OUT[0])

    # Create a ParameterList object
    # parameter_list = torch.nn.ParameterList(list_of_tensors)
    #parameter_list = torch.nn.ParameterList(list_of_tensors)
    if MAX_SOURCES_COUNT == 1:
      parameter_list = nn.ParameterList([*list_of_tensors[0]])
    elif MAX_SOURCES_COUNT == 2:
      parameter_list = nn.ParameterList([*list_of_tensors[0], *list_of_tensors[1]])
    elif MAX_SOURCES_COUNT == 3:
      parameter_list = nn.ParameterList([*list_of_tensors[0], *list_of_tensors[1], *list_of_tensors[2]])


    # Convert the ParameterList to a list of tensors
    #tensor_list = list(parameter_list)

    # Concatenate the tensors along the first dimension
    #concatenated_tensor = torch.cat(tensor_list, dim=0)

    s_wte.mid_learned_embedding = parameter_list #concatenated_tensor


    train_data_loaders, val_data_loaders, test_data_loaders = make_dataloaders( train,
                                                                                dev,
                                                                                test,
                                                                                train_label,
                                                                                dev_label,
                                                                                test_label )

    custom_train_data_loaders = make_custom_dataloader(X_train_text_custom, y_train_custom)

    print("Training set size:",   len(train))
    print("Dev set size:", len(dev))
    print("Test set size:",       len(test))
    print("Custom Training set size:",   len(y_train_custom))
    print("Stuffed tokens count:",  STUFF_COUNT)
    print("Seed set:",  SEED_SET)
    print('-'*10)
    print(BOUND_SUBSET)

    #del X_train_text, X_val_text, X_test_text, X_train_text_custom


    #print({k: v for k, v in sorted(SENTS_LENGHTS.items(), key=lambda item: item[1], reverse=True)})

    #model_hf = initilize_model(freeze_embedding=False)

    # s_wte = SoftEmbedding(model_hf.get_input_embeddings(),
    #                       initialize_from_vocab=True)

    #
    #s_wte.mid_learned_embedding = torch.load(f'mid_learned_embedding_{fold}_{predefined_seed_set[source_count - 1]}.pth')

    print('\n')

    #os.system(f'rm -rf mid_learned_embedding_{fold}_{predefined_seed_set[source_count - 1]}.pth')

    print('Loaded.')

    print('\n')

    model_hf.encoder.set_input_embeddings(s_wte)

    model = SentimentClassifier(model_hf)


    model = model.to(device)

    #

    print('\n')

    optimizer = load_optimizer(model, learning_rate=LEARNING_RATE, weight_decay=WD)

    BOUND_SUBSET[0] = len(y_train_custom)

    best_performance_test = train_eval(model,
                                      train_data_loaders,
                                      val_data_loaders,
                                      test_data_loaders,
                                      custom_train_data_loaders,
                                      len(train_label),
                                      len(dev_label),
                                      len(test_label),
                                      len(y_train_custom),
                                      fold,
                                      epochs=NUM_TRAIN_EPOCHS)

    fold_time = time.time() - start_time

    print(f'Time taken for fold {fold}: {fold_time} seconds')
    print(f'Best performance in test is {best_performance_test}')
    for key in folds_performance_test:
      folds_performance_test[key] += best_performance_test[key]
    print(f'End of fold {fold}.\n = = = = = = = = \n')


#
print(f'Result for {fold} folds in test set:')
for key in folds_performance_test:
  folds_performance_test[key] /= FOLDS
print(folds_performance_test)